###PHASE 3: BASELINE FRAUD DETECTION MODEL

Step 1 — Defining the Input Path

Before we load anything, we just define where the data lives. We store the file path in a variable called input_path

In [0]:
input_path = "dbfs:/Volumes/workspace/default/raw_data/streaming_output"
print(input_path)

dbfs:/Volumes/workspace/default/raw_data/streaming_output


Step 2 — Loading the Delta Table

Now we actually load the data. We use spark.read.format("delta").load() to read the Delta table as a regular batch DataFrame.

In [0]:
df = spark.read.format("delta").load(input_path)
print("Data loaded successfully")

Data loaded successfully


Step 3 — Printing the Schema

Now let's look at what columns we have and what data types they are. 

In [0]:
df.printSchema()

root
 |-- step: double (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: double (nullable = true)
 |-- is_high_risk_type: integer (nullable = true)
 |-- balance_diff: double (nullable = true)



Step 4 — Counting Total Rows and Fraud Cases

Now let's verify the data is complete and confirm the class imbalance

In [0]:
total = df.count()
fraud = df.filter(df.isFraud == 1).count()

print(f"Total transactions : {total}")
print(f"Fraud cases        : {fraud}")
print(f"Fraud rate         : {round(fraud / total * 100, 4)}%")

Total transactions : 49962
Fraud cases        : 69
Fraud rate         : 0.1381%


Step 5 — Showing Sample Rows

Before we do any modeling, we want to visually look at a few rows of actual data. This is called a sanity check — just making sure the data looks reasonable with our own eyes.

In [0]:
df.show(5, truncate=False)

+-----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+-----------------+------------+
|step |type    |amount   |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|is_high_risk_type|balance_diff|
+-----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+-----------------+------------+
|396.0|CASH_OUT|118326.41|C944557061 |121796.0     |3469.59       |C1487077526|1182256.11    |1300582.52    |0      |0.0           |1                |118326.41   |
|373.0|PAYMENT |17068.83 |C255803662 |20451.0      |3382.17       |M499251811 |0.0           |0.0           |0      |0.0           |0                |17068.83    |
|133.0|CASH_OUT|369264.57|C1339328459|124268.0     |0.0           |C1297375011|868843.17     |1238107.74    |0      |0.0           |1                |124268.0    |
|351.0|PAYMENT |

Step 6 — Selecting the Feature Columns

In [0]:
feature_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "is_high_risk_type",
    "balance_diff"
]

print("Feature columns selected:")
for col in feature_cols:
    print(f"  - {col}")

Feature columns selected:
  - amount
  - oldbalanceOrg
  - newbalanceOrig
  - oldbalanceDest
  - newbalanceDest
  - is_high_risk_type
  - balance_diff


Step 7 — Assembling Features into a Vector

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_assembled = assembler.transform(df)

print("Vector assembly complete")
print(f"New column count: {len(df_assembled.columns)}")
df_assembled.select("features", "isFraud").show(5, truncate=False)


Vector assembly complete
New column count: 14
+----------------------------------------------------------------+-------+
|features                                                        |isFraud|
+----------------------------------------------------------------+-------+
|[118326.41,121796.0,3469.59,1182256.11,1300582.52,1.0,118326.41]|0      |
|[17068.83,20451.0,3382.17,0.0,0.0,0.0,17068.83]                 |0      |
|[369264.57,124268.0,0.0,868843.17,1238107.74,1.0,124268.0]      |0      |
|(7,[0],[10039.08])                                              |0      |
|(7,[0],[7639.89])                                               |0      |
+----------------------------------------------------------------+-------+
only showing top 5 rows


Step 8 — Splitting Data into Train and Test Sets

In [0]:
train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows : {train_df.count()}")
print(f"Test rows     : {test_df.count()}")
print(f"Train fraud   : {train_df.filter(train_df.isFraud == 1).count()}")
print(f"Test fraud    : {test_df.filter(test_df.isFraud == 1).count()}")

Training rows : 39922
Test rows     : 10040
Train fraud   : 56
Test fraud    : 13


Step 9 — Calculating Class Weights

In [0]:
train_total = train_df.count()
train_fraud = train_df.filter(train_df.isFraud == 1).count()
train_nonfraud = train_total - train_fraud

weight_fraud = train_total / (2 * train_fraud)
weight_nonfraud = train_total / (2 * train_nonfraud)

print(f"Weight for fraud     : {round(weight_fraud, 4)}")
print(f"Weight for non-fraud : {round(weight_nonfraud, 4)}")

Weight for fraud     : 356.4464
Weight for non-fraud : 0.5007


Step 10 — Adding Class Weights to the Training DataFrame

In [0]:
from pyspark.sql.functions import when

train_weighted = train_df.withColumn(
    "classWeight",
    when(train_df.isFraud == 1, weight_fraud).otherwise(weight_nonfraud)
)

print("Class weights added")
train_weighted.select("isFraud", "classWeight").show(5)

Class weights added
+-------+------------------+
|isFraud|       classWeight|
+-------+------------------+
|      0|0.5007023528821553|
|      0|0.5007023528821553|
|      0|0.5007023528821553|
|      0|0.5007023528821553|
|      0|0.5007023528821553|
+-------+------------------+
only showing top 5 rows


Step 11 — Training the Random Forest Model

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="isFraud",
    weightCol="classWeight",
    numTrees=100,
    seed=42
)

print("Training started...")
rf_model = rf.fit(train_weighted)
print("Training complete")

Training started...
Training complete


Step 12 — Making Predictions on the Test Set

In [0]:
predictions = rf_model.transform(test_df)

print("Predictions made")
predictions.select("isFraud", "prediction", "probability").show(5, truncate=False)

Predictions made
+-------+----------+------------------------------------------+
|isFraud|prediction|probability                               |
+-------+----------+------------------------------------------+
|0      |0.0       |[0.9964125212322489,0.0035874787677511563]|
|0      |0.0       |[0.9996107562904117,3.8924370958830185E-4]|
|0      |0.0       |[0.9077460921961683,0.09225390780383169]  |
|0      |0.0       |[0.9996107562904117,3.8924370958830185E-4]|
|0      |0.0       |[0.9916264313804101,0.008373568619590011] |
+-------+----------+------------------------------------------+
only showing top 5 rows


Step 13 — Evaluating the Model

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

multi_eval = MulticlassClassificationEvaluator(
    labelCol="isFraud",
    predictionCol="prediction"
)

binary_eval = BinaryClassificationEvaluator(
    labelCol="isFraud",
    rawPredictionCol="rawPrediction"
)

precision = multi_eval.evaluate(predictions, {multi_eval.metricName: "precisionByLabel"})
recall    = multi_eval.evaluate(predictions, {multi_eval.metricName: "recallByLabel"})
f1        = multi_eval.evaluate(predictions, {multi_eval.metricName: "f1"})
auc       = binary_eval.evaluate(predictions)

print(f"Precision : {round(precision, 4)}")
print(f"Recall    : {round(recall, 4)}")
print(f"F1 Score  : {round(f1, 4)}")
print(f"AUC       : {round(auc, 4)}")

Precision : 0.9998
Recall    : 0.9907
F1 Score  : 0.9942
AUC       : 0.993


Step 14 — Saving the Trained Model


In [0]:
model_path = "dbfs:/Volumes/workspace/default/raw_data/fraud_model"

rf_model.save(model_path)

print(f"Model saved to: {model_path}")

Model saved to: dbfs:/Volumes/workspace/default/raw_data/fraud_model
